# Free keyword clustering tool: SERP overlap + embeddings hybrid

A free, Colab-ready keyword clustering notebook by **Lazarina Stoy, [MLforSEO](https://www.mlforseo.com/)**.

Keyword clustering tools usually pick one of two philosophies:

* **Semantic (embeddings) clustering** groups keywords whose *meaning* is close. It needs nothing but the keyword list, so it's free and fast, but it can't tell you whether Google treats two similar-sounding queries as the same search (for example *cold brew recipe* and *cold brew maker* mean related things but get very different results pages).
* **SERP-overlap clustering** groups keywords that share ranking URLs. Shared URLs are the clearest evidence that one page can rank for both, but you need SERP data for every keyword, which costs money, and keywords without SERP data are left out.

This notebook runs both, shows you where they disagree, and then combines them into a **hybrid** that uses SERP evidence where you have it and meaning where you don't. Each cluster gets a label (its highest-volume keyword), a most central keyword, a rough intent hint and volume totals, ready to paste into your keyword map.

**What you need**

* A keyword CSV with a `keyword` column (and ideally `volume`).
* Optional: the top 10 organic URLs per keyword, as a long CSV with `keyword`, `rank`, `url` columns. Most SEO tools and the DataForSEO SERP API can export this. Section 3 includes an optional DataForSEO fetcher (about $0.0006 per keyword in the standard queue, $0.002 in live mode, at the time of writing).
* No API key for the sample run. Everything runs on a CPU in a couple of minutes.

**Steps**

1. Settings and data
2. Method A: embeddings clustering
3. Method B: SERP-overlap clustering (with an optional DataForSEO fetcher)
4. Where the two methods disagree
5. Method C: the hybrid
6. Cluster names, intent hints and volume totals
7. Export (CSV + Excel file you can import into Google Sheets)

The sample data is a **synthetic** home-coffee keyword set: real-looking searches, invented volumes and made-up SERP URLs on fictional `.example` domains. See `sample_data/README.md`.

In [1]:
# Install anything that's missing (Colab already has most of these)
import importlib, importlib.util, subprocess, sys
for pkg, mod in [("sentence-transformers", "sentence_transformers"), ("scikit-learn", "sklearn"),
                 ("pandas", "pandas"), ("openpyxl", "openpyxl"), ("requests", "requests")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

## 1. Settings and data

Leave `USE_SAMPLE = True` for the first run. To use your own data, set it to `False` and upload your files to Colab (folder icon on the left), then change the paths.

Your keyword file needs a `keyword` column; `volume` is optional but makes the cluster labels much better. Your SERP file (optional) needs one row per ranking URL: `keyword`, `rank`, `url`.

In [2]:
USE_SAMPLE = True

KEYWORDS_CSV = "my_keywords.csv"      # your file: keyword, volume
SERP_CSV = "my_serp_top10.csv"        # optional: keyword, rank, url  (set to None if you have no SERP data)

# Tuning knobs (the defaults suit short keywords; section 2 and 5 explain each one)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DISTANCE_THRESHOLD = 0.40   # lower = tighter, more clusters
SERP_MIN_SHARED_URLS = 3              # how many top-10 URLs two keywords must share
SERP_TOP_N = 10                       # compare the top N organic URLs
HYBRID_SERP_WEIGHT = 0.6              # how much SERP evidence counts vs meaning (0 to 1)
HYBRID_DISTANCE_THRESHOLD = 0.60      # lower = tighter, more clusters

REPO_RAW = "https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/keyword-clustering-serp-embeddings/"


def read_sample(name):
    """Read a sample file locally if it's there (cloned repo), otherwise from GitHub (Colab)."""
    local = Path("sample_data") / name
    return pd.read_csv(local if local.exists() else REPO_RAW + "sample_data/" + name)


if USE_SAMPLE:
    keywords = read_sample("keywords.csv")
    serp = read_sample("serp_top10.csv")
else:
    keywords = pd.read_csv(KEYWORDS_CSV)
    serp = pd.read_csv(SERP_CSV) if SERP_CSV else pd.DataFrame(columns=["keyword", "rank", "url"])

# Clean up: lowercase, trim, merge duplicates (summing volume)
keywords.columns = [c.strip().lower() for c in keywords.columns]
if "volume" not in keywords.columns:
    keywords["volume"] = 0
keywords["keyword"] = keywords["keyword"].astype(str).str.lower().str.strip()
keywords["volume"] = pd.to_numeric(keywords["volume"], errors="coerce").fillna(0).astype(int)
keywords = (keywords[keywords["keyword"] != ""]
            .groupby("keyword", as_index=False)["volume"].sum()
            .sort_values("volume", ascending=False, ignore_index=True))

serp.columns = [c.strip().lower() for c in serp.columns]
serp["keyword"] = serp["keyword"].astype(str).str.lower().str.strip()

print(f"{len(keywords)} keywords, total volume {keywords['volume'].sum():,}")
print(f"SERP data for {serp['keyword'].nunique()} of them")
keywords.head(10)

160 keywords, total volume 125,780
SERP data for 157 of them


,keyword,volume
0,best espresso machine,18100
1,latte vs cappuccino,12100
2,best coffee beans,9900
3,best coffee grinder,9900
4,cold brew recipe,8100
5,best milk frother,6600
6,how much caffeine in espresso,6600
7,how to make pour over coffee,3600
8,how to descale espresso machine,2900
9,best decaf coffee,2400


## 2. Method A: embeddings clustering

Each keyword is turned into an [embedding](https://www.mlforseo.com/glossary-term/embedding/) (a list of numbers that captures its meaning) with `all-MiniLM-L6-v2`, a small, free [Sentence-BERT](https://www.mlforseo.com/glossary-term/sentence-bert-sbert/) model that runs fine on a CPU. Keywords are then grouped with [hierarchical (agglomerative) clustering](https://www.mlforseo.com/glossary-term/hierarchical-clustering/): it keeps merging the closest groups until the next merge would join groups that are further apart than `EMBEDDING_DISTANCE_THRESHOLD` (in [cosine](https://www.mlforseo.com/glossary-term/cosine-similarity/) distance, where 0 means identical meaning). You don't have to guess the number of clusters up front.

[HDBSCAN](https://www.mlforseo.com/glossary-term/hdbscan/) is included as an alternative. It finds dense groups and labels everything else as noise (`-1`). On short keyword lists it tends to either lump most keywords into a few huge groups or leave many as noise (you'll see this on the sample), which is why it isn't the default.

In [3]:
model = SentenceTransformer(EMBEDDING_MODEL, device="cpu")
kw_list = keywords["keyword"].tolist()
embeddings = model.encode(kw_list, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
emb_sim = embeddings @ embeddings.T          # cosine similarity matrix (vectors are normalised)

keywords["emb_cluster"] = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=EMBEDDING_DISTANCE_THRESHOLD,
    metric="cosine",
    linkage="average",
).fit_predict(embeddings)

# Alternative: HDBSCAN (min_cluster_size = smallest group you'd call a cluster)
keywords["emb_hdbscan"] = HDBSCAN(min_cluster_size=3, copy=True).fit_predict(embeddings)

print("Agglomerative:", keywords["emb_cluster"].nunique(), "clusters")
print("HDBSCAN:", keywords["emb_hdbscan"].nunique() - (keywords["emb_hdbscan"] == -1).any(),
      "clusters,", (keywords["emb_hdbscan"] == -1).sum(), "keywords left as noise")

# Peek at the three biggest embedding clusters
for cid in keywords["emb_cluster"].value_counts().index[:3]:
    print(f"\ncluster {cid}:", ", ".join(keywords.loc[keywords["emb_cluster"] == cid, "keyword"]))

Agglomerative: 62 clusters
HDBSCAN: 2 clusters, 15 keywords left as noise

cluster 9: best espresso machine, how to backflush espresso machine, how does an espresso machine work, top rated espresso machines, espresso machine black friday, espresso machine deals, what is a heat exchanger espresso machine, espresso machine reviews, best espresso machine for beginners, best budget espresso machine, best home espresso machine 2026, refurbished espresso machine, best espresso machine for home, cheap espresso machine, espresso machine sale, buy espresso machine, best espresso machine under 500, how to use an espresso machine, espresso machine parts explained

cluster 1: cold brew recipe, cold brew ratio, cold brew coffee to water ratio, cold brew vs iced coffee, best cold brew maker, how long does cold brew last in the fridge, cold brew coffee maker, does cold brew have more caffeine, cold brew pitcher, how to make cold brew at home, best cold brew system, cold brew vs nitro, easy cold brew 

## 3. Method B: SERP-overlap clustering

Two keywords are treated as "the same search" when their top 10 organic results share at least `SERP_MIN_SHARED_URLS` URLs. Thresholds of 3 or 4 are common in commercial tools; higher is stricter.

The grouping is a simple, explainable "hard" method (see [hard clustering](https://www.mlforseo.com/glossary-term/hard-clustering/)): take the highest-volume keyword that isn't in a cluster yet, put every unclustered keyword that shares enough URLs **with it** into its cluster, and repeat. Because every member is compared with the cluster's head keyword (and not with each other), clusters can't drift through long chains of loosely related keywords.

URLs are normalised first (protocol, `www.`, tracking parameters such as `utm_` and trailing slashes removed) so the same page is always counted once.

### Optional: fetch SERPs with DataForSEO

If you don't have a SERP export, set `FETCH_SERPS = True`. It calls the DataForSEO [Google Organic SERP API](https://docs.dataforseo.com/v3/serp/google/organic/live/advanced/) (v3, live mode, one keyword per request). Live mode costs about $0.002 per keyword (the standard queue is cheaper, about $0.0006, but you have to collect results later). For 500 keywords that's roughly $1. Your login and API password are read at runtime and never saved in the notebook.

In [4]:
FETCH_SERPS = False          # True = pull SERPs from DataForSEO for keywords missing from your SERP file
LOCATION_CODE = 2840         # 2840 = United States, 2826 = United Kingdom (use the market of your SERP file)
LANGUAGE_CODE = "en"


def get_secret(name):
    """Colab secrets first, then an environment variable, then a hidden prompt. Never hard-code keys."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    from getpass import getpass
    return getpass(f"{name}: ")


def fetch_serp_dataforseo(keyword, login, password):
    """Return [(rank, url), ...] for the organic top 10 of one keyword."""
    resp = requests.post(
        "https://api.dataforseo.com/v3/serp/google/organic/live/advanced",
        auth=(login, password),
        json=[{"keyword": keyword, "location_code": LOCATION_CODE,
               "language_code": LANGUAGE_CODE, "depth": SERP_TOP_N}],
        timeout=60,
    )
    if resp.status_code == 401:
        raise RuntimeError("DataForSEO rejected the login. Use your account email and the API password "
                           "from https://app.dataforseo.com/api-access (not your dashboard password).")
    resp.raise_for_status()
    data = resp.json()
    if data.get("status_code") != 20000:           # account-level problems (e.g. no balance)
        raise RuntimeError(f"DataForSEO {data.get('status_code')}: {data.get('status_message')}")
    task = data["tasks"][0]
    if task.get("status_code") != 20000:           # this keyword's task failed (bad parameter etc.)
        raise RuntimeError(f"DataForSEO task {task.get('status_code')}: {task.get('status_message')}")
    items = (task.get("result") or [{}])[0].get("items") or []
    return [(it["rank_group"], it["url"]) for it in items if it.get("type") == "organic"][:SERP_TOP_N]


if FETCH_SERPS and not USE_SAMPLE:
    login, password = get_secret("DATAFORSEO_LOGIN"), get_secret("DATAFORSEO_PASSWORD")
    missing = [k for k in kw_list if k not in set(serp["keyword"])]
    print(f"Fetching {len(missing)} SERPs (about ${len(missing) * 0.002:.2f} in live mode)")
    rows = []
    for k in missing:
        try:
            rows += [{"keyword": k, "rank": r, "url": u} for r, u in fetch_serp_dataforseo(k, login, password)]
        except Exception as e:
            print("  failed:", k, e)
            if "rejected the login" in str(e):
                break                                  # no point retrying every keyword with a bad login
    serp = pd.concat([serp, pd.DataFrame(rows)], ignore_index=True)
    serp.to_csv("serp_top10_fetched.csv", index=False)   # keep a copy so you never pay twice

In [5]:
TRACKING_PARAMS = re.compile(r"^(utm_[a-z]+|gclid|fbclid|msclkid|srsltid|ref|source)=", re.I)


def normalise_url(url):
    """Lowercase, drop protocol, www., fragments, tracking parameters and trailing slashes.
    Other query parameters are kept (e.g. a video's ?v= ID identifies a different page)."""
    url = str(url).strip()
    url = re.sub(r"^https?://", "", url, flags=re.I)
    url = re.sub(r"^www\.", "", url, flags=re.I)
    url = url.split("#")[0]
    base, _, query = url.partition("?")
    kept = [p for p in query.split("&") if p and not TRACKING_PARAMS.match(p)]
    return base.lower().rstrip("/") + ("?" + "&".join(kept) if kept else "")


serp["url_norm"] = serp["url"].map(normalise_url)
serp = serp.sort_values(["keyword", "rank"])
url_sets = (serp.groupby("keyword")["url_norm"]
            .apply(lambda s: set(s.head(SERP_TOP_N))).to_dict())

has_serp = np.array([k in url_sets for k in kw_list])
n = len(kw_list)
shared = np.zeros((n, n), dtype=int)       # shared[i, j] = URLs keywords i and j have in common
idx = [i for i in range(n) if has_serp[i]]
for a in idx:
    for b in idx:
        if b >= a:
            shared[a, b] = shared[b, a] = len(url_sets[kw_list[a]] & url_sets[kw_list[b]])


def serp_overlap_clusters(min_shared):
    """Hard clustering: highest-volume unclustered keyword becomes a seed, takes everything that
    shares >= min_shared URLs with it. Keywords without SERP data get -1."""
    labels = np.full(n, -1)
    next_id = 0
    for seed in np.argsort(-keywords["volume"].values, kind="stable"):
        if labels[seed] != -1 or not has_serp[seed]:
            continue
        members = [j for j in idx if labels[j] == -1 and shared[seed, j] >= min_shared]
        labels[members] = next_id
        labels[seed] = next_id
        next_id += 1
    return labels


keywords["serp_cluster"] = serp_overlap_clusters(SERP_MIN_SHARED_URLS)
print(keywords.loc[has_serp, "serp_cluster"].nunique(), "SERP clusters;",
      (~has_serp).sum(), "keywords have no SERP data (left out of this method)")

for cid in keywords["serp_cluster"].value_counts().drop(-1, errors="ignore").index[:3]:
    print(f"\ncluster {cid}:", ", ".join(keywords.loc[keywords["serp_cluster"] == cid, "keyword"]))

35 SERP clusters; 3 keywords have no SERP data (left out of this method)

cluster 0: best espresso machine, best coffee beans for espresso, top rated espresso machines, best espresso grinder, espresso machine reviews, best espresso machine for beginners, best budget espresso machine, best home espresso machine 2026, best espresso machine for home, best espresso machine under 500

cluster 16: how to dial in espresso grind, coffee grind size chart, espresso tastes sour, grind size for espresso, espresso machine not pumping water, grind size for pour over, grind size for french press, how fine to grind coffee for moka pot, espresso shot running too fast

cluster 4: cold brew recipe, cold brew ratio, cold brew coffee to water ratio, grind size for cold brew, how to make cold brew at home, easy cold brew recipe, cold brew concentrate recipe, how long to steep cold brew


## 4. Where the two methods disagree

The **adjusted Rand index** (ARI) scores how similar two clusterings are: 1 means identical, 0 means no better than chance.

The two tables below are the useful part:

* **Similar meaning, different SERPs**: pairs the embeddings put together that share almost no URLs. Usually a different intent (a recipe vs a product), so they likely need separate pages.
* **Different wording, same SERPs**: pairs Google answers with the same pages even though the words differ. One page can probably target both.

In [6]:
both = has_serp
ari = adjusted_rand_score(keywords.loc[both, "emb_cluster"], keywords.loc[both, "serp_cluster"])
print(f"Agreement between embeddings and SERP overlap (ARI): {ari:.2f}")

pairs = []
for a in idx:
    for b in idx:
        if b > a:
            pairs.append((kw_list[a], kw_list[b], round(float(emb_sim[a, b]), 2), shared[a, b]))
pairs = pd.DataFrame(pairs, columns=["keyword_a", "keyword_b", "meaning_similarity", "shared_urls"])

print("\nSimilar meaning, different SERPs (likely different intent):")
display(pairs[(pairs.meaning_similarity >= 0.70) & (pairs.shared_urls <= 1)]
        .sort_values("meaning_similarity", ascending=False).head(10))

print("Different wording, same SERPs (one page can likely rank for both):")
display(pairs[(pairs.meaning_similarity < 0.50) & (pairs.shared_urls >= SERP_MIN_SHARED_URLS)]
        .sort_values("shared_urls", ascending=False).head(10))

Agreement between embeddings and SERP overlap (ARI): 0.37

Similar meaning, different SERPs (likely different intent):


,keyword_a,keyword_b,meaning_similarity,shared_urls
8717,best oat milk for frothing,best milk frother for oat milk,0.97,0
9934,best budget espresso machine,cheap espresso machine,0.93,0
122,best espresso machine,cheap espresso machine,0.90,0
61,best espresso machine,espresso machine deals,0.89,1
11392,best espresso machine for home,cheap espresso machine,0.89,1
10672,best home espresso machine 2026,cheap espresso machine,0.88,1
9656,best espresso machine for beginners,cheap espresso machine,0.87,0
7806,espresso machine deals,best budget espresso machine,0.87,1
136,best espresso machine,buy espresso machine,0.87,0
12058,buy espresso machine,how to use an espresso machine,0.86,0


Different wording, same SERPs (one page can likely rank for both):


,keyword_a,keyword_b,meaning_similarity,shared_urls
8276,conical vs flat burr,stepped vs stepless grinder,0.35,8
12237,pour over bloom time,v60 recipe,0.22,8
3731,flat white vs latte,what is a cortado,0.13,8
1217,how to make pour over coffee,v60 recipe,0.19,8
11092,third wave water review,does water quality affect coffee taste,0.31,8
5982,washed vs natural process coffee,ethiopian coffee flavor profile,0.33,8
10989,microfoam vs macrofoam,how to froth milk without a frother,0.04,8
5571,chemex recipe,pour over bloom time,0.24,7
4566,are blade grinders bad,stepped vs stepless grinder,0.44,7
4752,arabica vs robusta,washed vs natural process coffee,0.10,7


## 5. Method C: the hybrid

The hybrid blends both signals into one similarity score for every pair of keywords:

`hybrid similarity = HYBRID_SERP_WEIGHT × (shared URLs ÷ 10) + (1 − HYBRID_SERP_WEIGHT) × meaning similarity`

* When both keywords have SERP data, Google's evidence counts for 60% by default and meaning for 40%.
* When either keyword has no SERP data, the score falls back to meaning similarity alone, so nothing is dropped.

The pairs are then clustered with the same agglomerative method as section 2, stopping at `HYBRID_DISTANCE_THRESHOLD`. Raise `HYBRID_SERP_WEIGHT` if you trust your SERP data and want stricter, intent-led clusters; lower it for broader, topic-led clusters.

In [7]:
serp_sim = shared / SERP_TOP_N
both_have_serp = np.outer(has_serp, has_serp)
hybrid_sim = np.where(both_have_serp,
                      HYBRID_SERP_WEIGHT * serp_sim + (1 - HYBRID_SERP_WEIGHT) * emb_sim,
                      emb_sim)
np.fill_diagonal(hybrid_sim, 1.0)
hybrid_dist = np.clip(1 - hybrid_sim, 0, None)

keywords["cluster_id"] = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=HYBRID_DISTANCE_THRESHOLD,
    metric="precomputed",
    linkage="average",
).fit_predict(hybrid_dist)

print(keywords["cluster_id"].nunique(), "hybrid clusters")
print("Agreement with SERP clusters (ARI):",
      round(adjusted_rand_score(keywords.loc[both, "serp_cluster"], keywords.loc[both, "cluster_id"]), 2))
print("Agreement with embedding clusters (ARI):",
      round(adjusted_rand_score(keywords["emb_cluster"], keywords["cluster_id"]), 2))

39 hybrid clusters
Agreement with SERP clusters (ARI): 0.79
Agreement with embedding clusters (ARI): 0.46


## 6. Cluster names, intent hints and volume totals

For each hybrid cluster:

* **cluster_label** is the highest-volume keyword (the one you'd usually make the page's primary keyword). With no volumes, it falls back to the most central keyword.
* **central_keyword** is the keyword closest in meaning to all the others (highest average similarity within the cluster). When it differs from the label, it's often a better H1 or page angle.
* **intent_hint** is a volume-weighted vote from simple English [search intent](https://www.mlforseo.com/glossary-term/search-intent/) rules (informational, comparison, commercial, transactional). It's a hint for a human to check, not a classifier. Edit the patterns for your market and language.

In [8]:
INTENT_RULES = [   # first match wins; edit for your niche and language
    ("transactional", r"\b(buy|price|prices|pricing|deal|deals|sale|cheap|discount|coupon|order|shop|refurbished|black friday|download|free trial|login|sign up)\b"),
    ("commercial",    r"\b(best|top|top rated|review|reviews|recommended|kit|subscription|tool|tools|software|app|apps|api|platform|generator|checker|tracker|alternative|alternatives|free)\b"),
    ("comparison",    r"\b(vs|versus|or|compared|difference|better than)\b"),
    ("informational", r"\b(how|what|why|when|which|does|do|is|are|can|should|guide|tips|recipe|ratio|chart|explained|tutorial)\b"),
]


def intent_of(keyword):
    for label, pattern in INTENT_RULES:
        if re.search(pattern, keyword):
            return label
    return "unclear"


keywords["intent_hint"] = keywords["keyword"].map(intent_of)
pos = {k: i for i, k in enumerate(kw_list)}

summaries = []
for cid, group in keywords.groupby("cluster_id"):
    ids = [pos[k] for k in group["keyword"]]
    centrality = hybrid_sim[np.ix_(ids, ids)].mean(axis=1)
    central = group["keyword"].iloc[int(np.argmax(centrality))]
    label = group.sort_values("volume", ascending=False)["keyword"].iloc[0] if group["volume"].sum() > 0 else central
    weighted_intent = (group.assign(w=group["volume"].clip(lower=1))
                       .groupby("intent_hint")["w"].sum().sort_values(ascending=False))
    summaries.append({
        "cluster_id": cid,
        "cluster_label": label,
        "central_keyword": central,
        "keywords": len(group),
        "total_volume": int(group["volume"].sum()),
        "intent_hint": weighted_intent.index[0],
        "intent_mix": ", ".join(f"{k} {v / weighted_intent.sum():.0%}" for k, v in weighted_intent.items()),
        "keywords_without_serp": int((~has_serp[ids]).sum()),
        "members": " | ".join(group.sort_values("volume", ascending=False)["keyword"]),
    })

clusters = (pd.DataFrame(summaries)
            .sort_values("total_volume", ascending=False, ignore_index=True))
keywords = keywords.merge(clusters[["cluster_id", "cluster_label", "central_keyword"]], on="cluster_id")
keywords["is_primary"] = keywords["keyword"] == keywords["cluster_label"]
keywords["has_serp_data"] = keywords["keyword"].isin(url_sets.keys())

clusters.drop(columns="members").head(15)

,cluster_id,cluster_label,central_keyword,keywords,total_volume,intent_hint,intent_mix,keywords_without_serp
0,18,best espresso machine,top rated espresso machines,9,19600,commercial,commercial 100%,1
1,0,latte vs cappuccino,flat white vs latte,4,13450,comparison,"comparison 94%, informational 6%",0
2,8,best coffee beans,best coffee beans,6,10960,commercial,commercial 100%,0
3,17,best coffee grinder,best manual coffee grinder,7,10910,commercial,commercial 100%,1
4,23,cold brew recipe,easy cold brew recipe,7,9880,informational,informational 100%,0
5,16,best milk frother,best milk frother for oat milk,4,7610,commercial,"commercial 89%, unclear 11%",0
6,32,how much caffeine in espresso,how much caffeine in a cup of coffee,3,6930,informational,informational 100%,0
7,1,how to make pour over coffee,best pour over recipe,7,4800,informational,"informational 84%, comparison 10%, commercial 4%, unclear 2%",0
8,6,how to descale espresso machine,best descaler for espresso machine,6,3640,informational,"informational 89%, unclear 7%, commercial 4%, comparison 0%",0
9,9,how to dial in espresso grind,coffee grind size chart,7,3190,informational,"informational 67%, unclear 33%",0


In [9]:
# A quick look at the biggest clusters with their members
for _, row in clusters.head(5).iterrows():
    print(f"[{row.cluster_label}]  {row.keywords} keywords, {row.total_volume:,} searches, {row.intent_hint}")
    print("   ", row.members, "\n")

[best espresso machine]  9 keywords, 19,600 searches, commercial
    best espresso machine | top rated espresso machines | nespresso aeroccino review | espresso machine reviews | best espresso machine for beginners | best budget espresso machine | best home espresso machine 2026 | best espresso machine for home | best espresso machine under 500 

[latte vs cappuccino]  4 keywords, 13,450 searches, comparison
    latte vs cappuccino | flat white vs latte | what is a cortado | what is a gibraltar coffee 

[best coffee beans]  6 keywords, 10,960 searches, commercial
    best coffee beans | best coffee beans for espresso | best whole bean coffee | best coffee beans for cold brew | best coffee beans for french press | best coffee subscription 

[best coffee grinder]  7 keywords, 10,910 searches, commercial
    best coffee grinder | best burr grinder | best espresso grinder | coffee grinder reviews | best coffee grinder for pour over | best budget burr grinder | best manual coffee grinder 



## 7. Export

You get three files:

* `keyword_clusters.csv`: one row per keyword with its hybrid cluster, label, primary flag, intent hint and the cluster IDs from each individual method (so you can see where they differed).
* `cluster_summary.csv`: one row per cluster with label, central keyword, size, total volume and intent mix.
* `keyword_clusters.xlsx`: both tables as tabs. In Google Sheets, use **File → Import → Upload** to open it as a sheet for your keyword map.

In [10]:
keyword_out = keywords[["cluster_id", "cluster_label", "keyword", "volume", "is_primary", "intent_hint",
                        "central_keyword", "has_serp_data", "serp_cluster", "emb_cluster"]].copy()
# biggest clusters first, highest-volume keyword first within each cluster
keyword_out["cluster_rank"] = keyword_out["cluster_id"].map({cid: i for i, cid in enumerate(clusters["cluster_id"])})
keyword_out = (keyword_out.sort_values(["cluster_rank", "volume"], ascending=[True, False])
               .drop(columns="cluster_rank"))

keyword_out.to_csv("keyword_clusters.csv", index=False)
clusters.to_csv("cluster_summary.csv", index=False)
with pd.ExcelWriter("keyword_clusters.xlsx") as xl:
    clusters.to_excel(xl, sheet_name="Clusters", index=False)
    keyword_out.to_excel(xl, sheet_name="Keywords", index=False)

print("Saved keyword_clusters.csv, cluster_summary.csv and keyword_clusters.xlsx")
# In Colab, download with:  from google.colab import files; files.download("keyword_clusters.xlsx")

Saved keyword_clusters.csv, cluster_summary.csv and keyword_clusters.xlsx


## How to read the results (and the limits)

* **Start with the summary.** Each hybrid cluster is a candidate page. The label is the primary keyword; the rest are secondary keywords and subheading ideas.
* **Check the mixed-intent clusters.** If `intent_mix` is split (say 50% commercial, 50% informational), look at the members. You may want two pages.
* **Use the disagreement tables** (section 4) for decisions about merging or splitting pages. They're the cases where meaning alone would have misled you.
* **Watch keywords without SERP data.** They are placed by meaning alone, so they can land in a cluster with a different intent. In the sample, *nespresso aeroccino review* (a milk frother) has no SERP data and joins the espresso machine reviews cluster because the words look alike. `keywords_without_serp` in the summary tells you where to look.
* **Tune, then re-run.** Too many tiny clusters: raise the distance thresholds. Clusters too broad: lower them, or raise `HYBRID_SERP_WEIGHT`.

**Limits, honestly:**

* SERPs change with location, device, personalisation and time. Overlap from a single snapshot is a strong signal, not a fact. Re-pull before big decisions.
* `all-MiniLM-L6-v2` is an English model. For other languages swap in a multilingual one such as `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
* The thresholds were tuned on the synthetic sample (short, English keywords). Your data may need different values, so eyeball a few clusters before you trust them.
* The intent hint is a handful of regular expressions. It's there to speed up a human review.
* The sample data is synthetic (invented volumes, fictional `.example` URLs), so the results show how the method behaves, not how any real SERP looks.

## What to do next

* Read the companion guide: [How to do keyword clustering with KeyBERT](https://www.mlforseo.com/machine-learning-implementation-guides/keyword-research/how-to-do-keyword-clustering-with-keybert/) and [What ML clustering is and where it fits in SEO](https://www.mlforseo.com/machine-learning-tutorials/topic-modeling-clustering/what-ml-clustering-is-and-where-it-fits-in-seo/).
* Learn how clustering works (and which model to pick when) in the [Introduction to Machine Learning for SEO course](https://www.mlforseo.com/courses/introduction-to-machine-learning-for-seo/).
* Browse more free notebooks in the [MLforSEO resources library](https://www.mlforseo.com/resources-library/) and the [mlforseo-code repo](https://github.com/lazarinastoy/mlforseo-code).

*Made by Lazarina Stoy, MLforSEO. MIT licence.*